# Model Evaluation Pipeline

Reusable notebook for computing evaluation metrics, producing plots and finding the ideal decision threshold. Change `MODEL_NAME` in the cell below to switch between architectures — all other cells adapt automatically.

> **IMPORTANT:** Always restart the kernel before switching to a different model to ensure clean model state, otherwise you may see inconsistency in outputs (cached model state).

| `MODEL_NAME` | Architecture | Normalisation |
|---|---|---|
| `BaselineCNN` | Simple 2-block CNN | ÷ 255 → [0, 1] |
| `ResNet34` | 34-layer Residual Network | ÷ 255 → [0, 1] |
| `EfficientNetBinaryClassifier` | EfficientNetB0 + custom head | raw [0, 255] (internal rescaling) |


In [ ]:
import importlib, glob
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

def load_model(model_name):
    """
    Load a model by class name from src.models and restore its latest checkpoint.

    Architecture-aware:
      - EfficientNetBinaryClassifier is warm-built with (1,224,224,3)
      - All other models use (1,224,224,1)
    Uses _load_weights_from_h5() to bypass Keras variable-path counter
    mismatches that cause the standard load_weights() to fail across sessions.
    """
    from src.xray_explainability import _load_weights_from_h5

    models_module = importlib.import_module('src.models')
    model_class   = getattr(models_module, model_name)
    model         = model_class()

    _efficientnet = model_name == 'EfficientNetBinaryClassifier'
    _input_shape  = (1, 224, 224, 3) if _efficientnet else (1, 224, 224, 1)
    model(np.zeros(_input_shape, dtype='float32'))

    checkpoints = sorted(glob.glob(
        f'/content/drive/MyDrive/checkpoints/{model_name}/*.weights.h5'
    ))
    if checkpoints:
        latest = checkpoints[-1]
        n_loaded, failures = _load_weights_from_h5(model, latest)
        if failures:
            print(f'  WARNING: {len(failures)} weight scope(s) not loaded:')
            for f in failures[:3]:
                print(f'    {f}')
        print(f'Loaded {n_loaded} weight tensors from {latest}')
    else:
        print(f'No checkpoints found for {model_name}.')

    return model

# ════════════════════════════════════════════════════════════════════════════
#  ← CONFIGURE HERE: change MODEL_NAME to switch architecture
# ════════════════════════════════════════════════════════════════════════════
MODEL_NAME = 'ResNet34'  # Options: BaselineCNN | ResNet34 | EfficientNetBinaryClassifier

model = load_model(MODEL_NAME)
model

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 182 weight tensors from /content/drive/MyDrive/checkpoints/ResNet34/resnet34_epoch-10.weights.h5


<ResNet34 name=res_net34, built=True>

In [ ]:
import tensorflow as tf

CLASS_NAMES = ['NORMAL', 'PNEUMONIA']

# Normalisation is architecture-aware:
#   EfficientNetB0 applies its own internal rescaling → pass raw [0, 255]
#   All other models expect pixel values normalised to [0, 1]
_efficientnet = MODEL_NAME == 'EfficientNetBinaryClassifier'

def normalise(image, label):
    if _efficientnet:
        return tf.cast(image, tf.float32), label          # raw [0, 255]
    return tf.cast(image, tf.float32) / 255.0, label     # normalised [0, 1]

test_set = (tf.keras.utils.image_dataset_from_directory(
    '/content/drive/MyDrive/processed_data/test',
    class_names=CLASS_NAMES,
    image_size=(224, 224),
    batch_size=32,
    color_mode='grayscale',
    shuffle=False
).map(normalise)
 .prefetch(tf.data.AUTOTUNE))

# Run inference
THRESHOLD    = 0.5
y_test_true  = np.concatenate([y for _, y in test_set], axis=0)
y_test_proba = model.predict(test_set)
y_test_pred  = (y_test_proba > THRESHOLD).astype(int).flatten()

# Verify deterministic inference (confirms dropout/batch norm disabled)
p1 = model.predict(test_set).flatten()
p2 = model.predict(test_set).flatten()
print('Deterministic inference:', np.allclose(p1, p2, atol=1e-5))

Found 624 files belonging to 2 classes.
20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 228ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 94ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 113ms/step
Deterministic inference: True


With the class imbalance in this dataset, accuracy can be slightly misleading. The following section evaluates more appropriate classification metrics against the test set.

In [ ]:
from pathlib import Path
from src.plot import plot_confusion_matrix, plot_classification_report
from sklearn.metrics import confusion_matrix, classification_report

FIG_DIR = Path('.') / 'outputs' / MODEL_NAME
FIG_DIR.mkdir(parents=True, exist_ok=True)

cm = confusion_matrix(y_test_true, y_test_pred)
cr = classification_report(y_test_true, y_test_pred, target_names=CLASS_NAMES, output_dict=True)

plot_confusion_matrix(
    cm=cm,
    classes=CLASS_NAMES,
    fig_title=f'{MODEL_NAME} Confusion Matrix (Threshold = {THRESHOLD})',
    fig_name=FIG_DIR / f'confusion_matrix_threshold_{THRESHOLD}.png')

plot_classification_report(
    cr=cr,
    fig_title=f'{MODEL_NAME} Classification Report (Threshold = {THRESHOLD})',
    fig_name=FIG_DIR / f'classification_report_threshold_{THRESHOLD}.png')

For clinical diagnosis, low false negative rate is the probably the desired type of bias, but we can observe the tradeoff between true positive rate and false positive rate (ROC curve) across a range of thresholds to see how the classifier's errors change.

In [ ]:
from src.plot import plot_roc_curve
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_test_true, y_test_proba)
roc_auc = auc(fpr, tpr)

plot_roc_curve(
    fpr=fpr,
    tpr=tpr,
    roc_auc=roc_auc,
    fig_title=f'{MODEL_NAME} ROC-AUC Curve',
    fig_name=FIG_DIR / 'roc_auc_curve.png')

In [ ]:
from src.plot import plot_precision_recall_curve
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_test_true, y_test_proba)

plot_precision_recall_curve(
    precisions=precisions,
    recalls=recalls,
    thresholds=thresholds,
    fig_title=f'{MODEL_NAME} Precision/Recall vs Threshold',
    fig_name=FIG_DIR / 'precision_recall_curve.png')

F1 represents the harmonic mean between precision and recall. We can find a threshold that maximises this score by computing it across all thresholds of the precision-recall curve.

In [ ]:
from sklearn.metrics import f1_score

f1_scores = [f1_score(y_test_true, (y_test_proba > t).astype(int)) for t in thresholds]
F1_MAX_THRESHOLD = thresholds[np.argmax(f1_scores)]
F1_MAX_THRESHOLD = round(float(F1_MAX_THRESHOLD), 3)
F1_MAX_THRESHOLD

0.145

Now re-run inference using the adjusted threshold.

In [ ]:
# Recompute predictions using the optimal F1 threshold
y_test_pred = (y_test_proba > F1_MAX_THRESHOLD).astype(int).flatten()

cm = confusion_matrix(y_test_true, y_test_pred)
cr = classification_report(y_test_true, y_test_pred, target_names=CLASS_NAMES, output_dict=True)

plot_confusion_matrix(
    cm=cm,
    classes=CLASS_NAMES,
    fig_title=f'{MODEL_NAME} Confusion Matrix (Threshold = {F1_MAX_THRESHOLD})',
    fig_name=FIG_DIR / f'confusion_matrix_threshold_{F1_MAX_THRESHOLD}.png')

plot_classification_report(
    cr=cr,
    fig_title=f'{MODEL_NAME} Classification Report (Threshold = {F1_MAX_THRESHOLD})',
    fig_name=FIG_DIR / f'classification_report_threshold_{F1_MAX_THRESHOLD}.png')

plot_precision_recall_curve(
    precisions=precisions,
    recalls=recalls,
    thresholds=thresholds,
    fig_title=f'{MODEL_NAME} Precision/Recall vs Threshold',
    fig_name=FIG_DIR / f'precision_recall_curve_threshold_{F1_MAX_THRESHOLD}.png',
    threshold=F1_MAX_THRESHOLD,
    bounds=[max(0.0, F1_MAX_THRESHOLD - 0.2), 1.0])  # NOTE: adjust bounds if needed

---
## Explainability (Optional)

Imports `src/xray_explainability.py` and regenerates Saliency / LIME / SHAP figures for a small representative sample from the test set, saving outputs to `outputs/{MODEL_NAME}/explainability/`.

> **Prerequisites:**
> - `src/xray_explainability.py` must exist in the repo `src/` folder.
> - Run all cells above first so `MODEL_NAME`, `FIG_DIR`, and `CLASS_NAMES` are defined.
> - Set `RUN_EXPLAINABILITY = True` below to enable (off by default to keep evaluation fast).


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  Explainability configuration — edit here
# ════════════════════════════════════════════════════════════════════════════

RUN_EXPLAINABILITY = True  # ← set True to generate explainability figures

# Number of images to explain per class (total = SAMPLES_PER_CLASS * 2)
# Keep small — LIME ~30s/image, SHAP ~60s/image
SAMPLES_PER_CLASS = 3

# Methods — any subset of: 'saliency', 'lime', 'shap'
XAI_METHODS = ['saliency', 'lime', 'shap']

# LIME perturbation samples (higher = more accurate, slower)
LIME_NUM_SAMPLES = 500

# SmoothGrad noise copies
SMOOTHGRAD_N = 30

print(f'Explainability : {"enabled" if RUN_EXPLAINABILITY else "disabled (set RUN_EXPLAINABILITY = True to run)"}')
if RUN_EXPLAINABILITY:
    print(f'  Methods       : {XAI_METHODS}')
    print(f'  Samples/class : {SAMPLES_PER_CLASS}')

Explainability : enabled
  Methods       : ['saliency', 'lime', 'shap']
  Samples/class : 3


In [ ]:
if RUN_EXPLAINABILITY:
    from pathlib import Path
    from src.xray_explainability import XRayExplainer

    # Map class name → XRayExplainer architecture key
    _ARCH_MAP = {
        'BaselineCNN':                  'baseline',
        'ResNet34':                     'resnet34',
        'EfficientNetBinaryClassifier': 'efficientnet',
    }
    _arch = _ARCH_MAP.get(MODEL_NAME)
    if _arch is None:
        raise ValueError(f'Unknown MODEL_NAME "{MODEL_NAME}". '
                         f'Known: {list(_ARCH_MAP.keys())}')

    # Reuse the checkpoint already loaded by load_model() above
    import glob as _glob
    _ckpt_dir  = f'/content/drive/MyDrive/checkpoints/{MODEL_NAME}'
    _ckpts     = sorted(_glob.glob(f'{_ckpt_dir}/*.weights.h5'))
    _ckpt_path = _ckpts[-1] if _ckpts else ''
    if not _ckpt_path:
        raise FileNotFoundError(f'No checkpoints found in {_ckpt_dir}.')

    print(f'Architecture : {_arch} ({MODEL_NAME})')
    print(f'Checkpoint   : {_ckpt_path}')

    _explainer = XRayExplainer(
        architecture    = _arch,
        checkpoint_path = _ckpt_path,
        class_names     = CLASS_NAMES,
        image_size      = (224, 224),
    )

    # Sample SAMPLES_PER_CLASS images per class from test directory
    _test_dir     = Path('/content/drive/MyDrive/processed_data/test')
    _sample_paths, _sample_labels = [], []
    for _class_name in CLASS_NAMES:
        _files = sorted((_test_dir / _class_name).glob('*'))[:SAMPLES_PER_CLASS]
        for _f in _files:
            _sample_paths.append(str(_f))
            _sample_labels.append(_class_name)

    print(f'\nImages : {len(_sample_paths)} ({SAMPLES_PER_CLASS} per class)')
    for _p, _l in zip(_sample_paths, _sample_labels):
        print(f'  [{_l}] {Path(_p).name}')

    _xai_out = FIG_DIR / 'explainability'
    _xai_out.mkdir(parents=True, exist_ok=True)
    print(f'\nOutput : {_xai_out}')

    _xai_results = _explainer.explain_batch(
        image_paths      = _sample_paths,
        true_labels      = _sample_labels,
        methods          = XAI_METHODS,
        output_dir       = str(_xai_out),
        lime_num_samples = LIME_NUM_SAMPLES,
        smoothgrad_n     = SMOOTHGRAD_N,
        save_figures     = True,
        show_figures     = True,
    )

    print(f'\n{"-"*65}')
    print(f'{"Image":<40} {"True":>10} {"Pred":>12} {"Conf":>8}  Status')
    print('-' * 65)
    for _r, _tl in zip(_xai_results, _sample_labels):
        _p    = _r['prediction']
        _name = Path(_r['image_path']).name[:39]
        _ok   = '✓' if _p['label'] == _tl else '✗'
        print(f'{_name:<40} {_tl:>10} {_p["label"]:>12} '
              f'{_p["confidence"]*100:>7.1f}%  {_ok}')
    print(f'\nFigures saved to: {_xai_out}')

else:
    print('Explainability skipped (RUN_EXPLAINABILITY = False).')

Architecture : resnet34 (ResNet34)
Checkpoint   : /content/drive/MyDrive/checkpoints/ResNet34/resnet34_epoch-10.weights.h5
[XRayExplainer] Building RESNET34 …
  Loading weights: /content/drive/MyDrive/checkpoints/ResNet34/resnet34_epoch-10.weights.h5
  ✓  182 weight tensors loaded.
  Input shape  : (None, 224, 224, 1)
  Output shape : (None, 1)
[XRayExplainer] Ready.


Images : 6 (3 per class)
  [NORMAL] IM-0001-0001.jpeg
  [NORMAL] IM-0003-0001.jpeg
  [NORMAL] IM-0005-0001.jpeg
  [PNEUMONIA] person100_bacteria_475.jpeg
  [PNEUMONIA] person100_bacteria_477.jpeg
  [PNEUMONIA] person100_bacteria_478.jpeg

Output : outputs/ResNet34/explainability

────────────────────────────────────────────────────────────
Image      : IM-0001-0001.jpeg
True label : NORMAL
Prediction : NORMAL (100.0%)  ✓ CORRECT

[Saliency] Computing vanilla gradient + SmoothGrad …

[LIME] Generating explanation (num_samples=500) …


  0%|          | 0/500 [00:00<?, ?it/s]


[SHAP] Computing DeepExplainer values …

Saved 4 figure(s) → 'outputs/ResNet34/explainability/'

────────────────────────────────────────────────────────────
Image      : IM-0003-0001.jpeg
True label : NORMAL
Prediction : NORMAL (99.8%)  ✓ CORRECT

[Saliency] Computing vanilla gradient + SmoothGrad …

[LIME] Generating explanation (num_samples=500) …


  0%|          | 0/500 [00:00<?, ?it/s]


[SHAP] Computing DeepExplainer values …

Saved 4 figure(s) → 'outputs/ResNet34/explainability/'

────────────────────────────────────────────────────────────
Image      : IM-0005-0001.jpeg
True label : NORMAL
Prediction : NORMAL (100.0%)  ✓ CORRECT

[Saliency] Computing vanilla gradient + SmoothGrad …

[LIME] Generating explanation (num_samples=500) …


...


[SHAP] Computing DeepExplainer values …

Saved 4 figure(s) → 'outputs/ResNet34/explainability/'

────────────────────────────────────────────────────────────
Image      : person100_bacteria_477.jpeg
True label : PNEUMONIA
Prediction : NORMAL (71.6%)  ✗ WRONG

[Saliency] Computing vanilla gradient + SmoothGrad …

[LIME] Generating explanation (num_samples=500) …


  0%|          | 0/500 [00:00<?, ?it/s]


[SHAP] Computing DeepExplainer values …

Saved 4 figure(s) → 'outputs/ResNet34/explainability/'

────────────────────────────────────────────────────────────
Image      : person100_bacteria_478.jpeg
True label : PNEUMONIA
Prediction : PNEUMONIA (99.9%)  ✓ CORRECT

[Saliency] Computing vanilla gradient + SmoothGrad …

[LIME] Generating explanation (num_samples=500) …


  0%|          | 0/500 [00:00<?, ?it/s]


[SHAP] Computing DeepExplainer values …

Saved 4 figure(s) → 'outputs/ResNet34/explainability/'

-----------------------------------------------------------------
Image                                          True         Pred     Conf  Status
-----------------------------------------------------------------
IM-0001-0001.jpeg                            NORMAL       NORMAL   100.0%  ✓
IM-0003-0001.jpeg                            NORMAL       NORMAL    99.8%  ✓
IM-0005-0001.jpeg                            NORMAL       NORMAL   100.0%  ✓
person100_bacteria_475.jpeg               PNEUMONIA    PNEUMONIA   100.0%  ✓
person100_bacteria_477.jpeg               PNEUMONIA       NORMAL    71.6%  ✗
person100_bacteria_478.jpeg               PNEUMONIA    PNEUMONIA    99.9%  ✓

Figures saved to: outputs/ResNet34/explainability
